8-6.ipynb をGoogle colab上のGPUで実行

In [3]:
import os
print(os.getcwd())

/content


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!pip install numpy==1.23.5
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 97.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
albucore 0.0.23 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.23.5 which is incompatible.
xarray 2025.1.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
scikit-image 0.25.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
pymc 5.21.2 requires numpy>=1.25.0, but you have numpy 1.23.5 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


In [4]:
#埋め込みの読み込み
import numpy as np
from gensim.models import KeyedVectors

def load_pretrained_embeddings(embedding_path, vocab_limit=None):
    embeddings = []
    token2id = {}
    id2token = {}

    # GoogleNews-vectors-negative300.bin を読み込む
    word_vectors = KeyedVectors.load_word2vec_format(embedding_path, binary=True)

    embedding_dim = word_vectors.vector_size
    embeddings.append(np.zeros(embedding_dim))  # <PAD>トークン用ゼロベクトル
    token2id['<PAD>'] = 0
    id2token[0] = '<PAD>'

    for idx, word in enumerate(word_vectors.index_to_key):
        if vocab_limit and (idx >= vocab_limit):
            break
        vector = word_vectors[word]

        token_id = len(embeddings)
        token2id[word] = token_id
        id2token[token_id] = word
        embeddings.append(vector)

    embedding_matrix = np.vstack(embeddings) #各単語の埋め込みを縦方向に結合(ベクトル→行列)
    return embedding_matrix, token2id, id2token

# 使い方
embedding_path = 'drive/MyDrive/NLP100/8/GoogleNews-vectors-negative300.bin'
embedding_matrix, token2id, id2token = load_pretrained_embeddings(embedding_path, vocab_limit=50000)


In [6]:
#データセットの読み込み
import csv
import torch

def load_sst_data(file_path, token2id):
    dataset = []
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            text = row['sentence']
            label = int(row['label'])

            # 単語ごとにトークンIDに変換
            tokens = text.split()
            input_ids = [token2id[token] for token in tokens if token in token2id]

            # 全部消えたらスキップ
            if len(input_ids) == 0:
                continue

            item = {
                'text': text,
                'label': torch.tensor([float(label)]),
                'input_ids': torch.tensor(input_ids)
            }
            dataset.append(item)
    return dataset

# 使い方
train_file = 'drive/MyDrive/NLP100/8//SST-2/train.tsv'
dev_file = 'drive/MyDrive/NLP100/8//SST-2/dev.tsv'

train_data = load_sst_data(train_file, token2id)
dev_data = load_sst_data(dev_file, token2id)


In [7]:
#モデルの構築
import torch
import torch.nn as nn

class TextAverageLogisticRegression(nn.Module):
    def __init__(self, embedding_matrix):
        super(TextAverageLogisticRegression, self).__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        # 埋め込み層
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
        self.embedding.weight.requires_grad = False  # 事前学習済みなので凍結

        # ロジスティック回帰（線形層）
        self.linear = nn.Linear(embedding_dim, 1)

    def forward(self, input_ids):
        """
        input_ids: (バッチサイズ, シーケンス長)
        """
        embedded = self.embedding(input_ids)  # (バッチサイズ, シーケンス長, 埋め込み次元)

        # マスク：IDが0（PAD）なら無視
        mask = (input_ids != 0).unsqueeze(-1)  # (バッチサイズ, シーケンス長, 1)
        masked_embeddings = embedded * mask

        # 平均計算（マスクした上で）
        summed = masked_embeddings.sum(dim=1)  # (バッチサイズ, 埋め込み次元)
        lengths = mask.sum(dim=1)  # (バッチサイズ, 1)
        avg_embedded = summed / lengths.clamp(min=1)  # 長さ0回避

        logits = self.linear(avg_embedded)  # (バッチサイズ, 1)
        probs = torch.sigmoid(logits)  # (バッチサイズ, 1)
        return probs


In [8]:
#学習(パディング処理の追加)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# --- 1. データセットとデータローダー準備 ---
class SSTDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': item['input_ids'],
            'label': item['label']
        }

def collate_batch(batch):
    input_ids = [item['input_ids'] for item in batch]
    labels = torch.stack([item['label'] for item in batch])

    input_ids_padded = torch.nn.utils.rnn.pad_sequence(
        input_ids, batch_first=True, padding_value=0
    )

    return {
        'input_ids': input_ids_padded,
        'label': labels
    }

# --- 2. データローダー作成 ---
train_dataset = SSTDataset(train_data)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)

# --- 3. モデル準備 ---
model = TextAverageLogisticRegression(embedding_matrix)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# --- 4. 学習ループ ---
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids']  # (バッチサイズ, シーケンス長)
        labels = batch['label']         # (バッチサイズ, 1)

        preds = model(input_ids)        # 順伝播
        loss = criterion(preds, labels) # 損失計算

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # 正解率計算
        predicted = (preds >= 0.5).float()  # 閾値0.5でポジ/ネガ分類
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / len(train_loader)
    accuracy = correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")


Epoch [1/30], Loss: 0.5187, Accuracy: 0.7704
Epoch [2/30], Loss: 0.4406, Accuracy: 0.8099
Epoch [3/30], Loss: 0.4239, Accuracy: 0.8157
Epoch [4/30], Loss: 0.4166, Accuracy: 0.8175
Epoch [5/30], Loss: 0.4126, Accuracy: 0.8204
Epoch [6/30], Loss: 0.4101, Accuracy: 0.8210
Epoch [7/30], Loss: 0.4085, Accuracy: 0.8220
Epoch [8/30], Loss: 0.4072, Accuracy: 0.8227
Epoch [9/30], Loss: 0.4063, Accuracy: 0.8228
Epoch [10/30], Loss: 0.4057, Accuracy: 0.8227
Epoch [11/30], Loss: 0.4052, Accuracy: 0.8236
Epoch [12/30], Loss: 0.4048, Accuracy: 0.8233
Epoch [13/30], Loss: 0.4044, Accuracy: 0.8232
Epoch [14/30], Loss: 0.4042, Accuracy: 0.8235
Epoch [15/30], Loss: 0.4040, Accuracy: 0.8237
Epoch [16/30], Loss: 0.4038, Accuracy: 0.8244
Epoch [17/30], Loss: 0.4036, Accuracy: 0.8236
Epoch [18/30], Loss: 0.4035, Accuracy: 0.8236
Epoch [19/30], Loss: 0.4033, Accuracy: 0.8244
Epoch [20/30], Loss: 0.4032, Accuracy: 0.8242
Epoch [21/30], Loss: 0.4032, Accuracy: 0.8244
Epoch [22/30], Loss: 0.4031, Accuracy: 0.82

In [9]:
#予測
# --- 1. dev用データローダー作成 ---
dev_dataset = SSTDataset(dev_data)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)

# --- 2. 評価モードにして、推論 ---
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch['input_ids']  # (バッチサイズ, シーケンス長)
        labels = batch['label']         # (バッチサイズ, 1)

        preds = model(input_ids)

        predicted = (preds >= 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

# --- 3. 正解率出力 ---
accuracy = correct / total
print(f"Dev Accuracy: {accuracy:.4f}")


Dev Accuracy: 0.7821
